# Integrate Control Samples

### 0. Load libraries

In [ ]:
import os
import sys
import random
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", UserWarning)

# Append necessary paths for module imports
sys.path.append("/projects/site/pred/ihb-g-deco/USERS/adaml9/bioinfo-blueprint/src/python")

import random
import matplotlib as mpl
random.seed(10)

from pathlib import Path

import cupy as cp
import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import rapids_singlecell as rsc
import matplotlib.pyplot as plt
import seaborn
seaborn.reset_orig()
%matplotlib inline

sc.settings.figdir = "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/figures/Multiome"

In [ ]:
rsc.__version__

In [ ]:
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator

rmm.reinitialize(
    managed_memory=False,  # Allows oversubscription
    pool_allocator=True,  # default is False
    devices=0,  # GPU device IDs to register. By default registers only GPU 0.
)
cp.cuda.set_allocator(rmm_cupy_allocator)

### 0. Setup

In [ ]:
import configparser

def read_config():
  config_file = Path('/projects/site/pred/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/global_analysis/multiome/grn') / 'config.ini'

  config = configparser.ConfigParser()
  config.read(config_file)
  return config


config = read_config()

### 1. Load data

In [ ]:
data_dir = config.get("DEFAULT", "PROCESSED_DATA_DIR")
figures_dir = config.get("DEFAULT", "FIGURES_DIR")
sample_names = config["SAMPLES"]["SAMPLE_NAMES"].split(",")

In [ ]:
# Load individual AnnData objects and tag them with sample names
adatas = []
for sample in sample_names:
    anndata_path = config[sample]["ANNDATA_FILE"]
    
    print(f"Loading {sample} from {anndata_path}")
    adata = sc.read_h5ad(anndata_path)
    
    # Add sample identifier as a column in obs
    adata.obs["sample"] = sample
    
    adatas.append(adata)

# Concatenate all AnnData objects
adata = ad.concat(adatas, join="outer", label="sample", keys=sample_names, index_unique="-")
adata.X = adata.X.toarray()

In [ ]:
# Saving count data
adata.layers["counts"] = adata.X.copy()
adata.raw = adata.copy()

### 2. Doublet detection with Scrublet

In [ ]:
rsc.get.anndata_to_GPU(adata)

In [ ]:
rsc.pp.scrublet(adata, batch_key="sample")

### 2. Integrate samples

In [ ]:
rsc.get.anndata_to_CPU(adata)

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)

adata.layers["lognorm"] = adata.X.copy()

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3', batch_key = 'sample', n_top_genes=3000, layer='counts')

In [ ]:
rsc.get.anndata_to_GPU(adata)

In [ ]:
rsc.pp.scale(adata, max_value=10)
rsc.tl.pca(adata, use_highly_variable=True)

In [ ]:
%%time
rsc.pp.harmony_integrate(adata, key="sample", dtype=cp.float32)

In [ ]:
# joint PCA space
rsc.pp.neighbors(adata, use_rep="X_pca")
rsc.tl.umap(adata, key_added="X_umap_pca")

# integrated harmony space
rsc.pp.neighbors(adata, use_rep="X_pca_harmony")
rsc.tl.umap(adata, key_added="X_umap_harmony")

In [ ]:
rsc.get.anndata_to_CPU(adata)

In [ ]:
color = ["sample", "cell_type_initial", "predicted_doublet"]

for emb in ["pca", "harmony"]:
    sc.pl.embedding(
        adata,
        basis=f"X_umap_{emb}",
        color=color,
        size=10,
        wspace=0.4,
        title=[f"{col} ({emb})" for col in color],
    )

In [ ]:
# Remove doublets
adata = adata[adata.obs["predicted_doublet"]==False].copy()

In [ ]:
sc.pp.highly_variable_genes(adata, flavor='seurat_v3', batch_key = 'sample', n_top_genes=3000, layer='counts')

In [ ]:
adata.X = adata.layers["lognorm"].copy()

In [ ]:
rsc.get.anndata_to_GPU(adata)

In [ ]:
rsc.pp.scale(adata, max_value=10)
rsc.tl.pca(adata, use_highly_variable=True)

In [ ]:
%%time
rsc.pp.harmony_integrate(adata, key="sample", dtype=cp.float32)

In [ ]:
# joint PCA space
rsc.pp.neighbors(adata, use_rep="X_pca")
rsc.tl.umap(adata, key_added="X_umap_pca")

# integrated harmony space
rsc.pp.neighbors(adata, use_rep="X_pca_harmony")
rsc.tl.umap(adata, key_added="X_umap_harmony")

In [ ]:
rsc.get.anndata_to_CPU(adata)

In [ ]:
color = ["sample", "cell_type_initial", "predicted_doublet"]

for emb in ["pca", "harmony"]:
    sc.pl.embedding(
        adata,
        basis=f"X_umap_{emb}",
        color=color,
        size=10,
        wspace=0.4,
        title=[f"{col} ({emb})" for col in color],
    )

### 3. Align cell type annotations using CellHint (Teichmann Lab)

In [ ]:
import cellhint

In [ ]:
alignment = cellhint.harmonize(adata, 'sample', 'cell_type_initial', use_rep="X_pca_harmony")

In [ ]:
alignment

In [ ]:
alignment.relation.head(20)

In [ ]:
alignment.write(os.path.join(data_dir, "cellhint_alignment_control_multiome_v2.pkl"))

In [ ]:
alignment.reannotation

In [ ]:
adata.obs[['low_hierarchy', 'high_hierarchy']] = alignment.reannotation.loc[adata.obs_names, ['reannotation', 'group']]
adata.obs[['low_hierarchy', 'high_hierarchy']]

In [ ]:
member_mat = alignment.base_distance.to_meta(turn_binary = True)
member_mat.iloc[:5, :5]

In [ ]:
import seaborn as sns

sns.clustermap(member_mat, cmap="Reds", figsize=(20, 20))

In [ ]:
cellhint.treeplot(alignment, order_dataset = True)

In [ ]:
alignment.relation.to_csv(os.path.join(data_dir, 'cellhint_alignment_control_multiome_v2.csv'), sep = ',', index = False)

### 4. Assign merged cell type annotations 

In [ ]:
color = ["low_hierarchy"]

for emb in ["harmony"]:
    sc.pl.embedding(
        adata,
        basis=f"X_umap_{emb}",
        color=color,
        size=10,
        wspace=0.4,
        title=[f"{col} ({emb})" for col in color],
    )

In [ ]:
color = ["high_hierarchy"]

with plt.rc_context({"figure.figsize": (5, 7), 
                     "legend.fontsize": 10, 
                     "axes.titlesize": 10}):
    for emb in ["harmony"]:
        sc.pl.embedding(
            adata,
            basis=f"X_umap_{emb}",
            color=color,
            size=10,
            wspace=0.4,
            title=[f"{col} ({emb})" for col in color],
        )

In [ ]:
alignment.reannotation["reannotation"].unique()

In [ ]:
resolved_annotations = {
    # Proliferative axis
    "TA cells = TA cells = TA cells": "TA cells",
    "cycling TA cells = cycling TA cells = cycling TA cells": "cycling TA cells",
    "cycling cells = cycling cells = NONE": "cycling cells",
    "Stem cells = Stem cells = Stem cells": "Stem cells",
    
    # Absorptive lineage
    "Enterocytes = Enterocytes = Enterocytes": "Enterocytes",
    "Early enterocytes = Early enterocytes = Enterocytes (FABP1 high)": "Early Enterocytes",
    "UNRESOLVED = UNRESOLVED = Enterocytes (S100P high)": "Enterocytes",
    "BEST4+ cells = BEST4+ cells = UNRESOLVED": "BEST4+ cells",
    
    # Secretory lineage
    "Goblet cells = Goblet cells ∋ Goblet cells": "Goblet cells",
    "Goblet cells = Goblet cells ∋ Early EECs III": "Goblet cells",  # keep as Goblet (absorptive-secretory split not distinct)
    
    # EEC progenitors
    "Early EECs = Early EECs = Early EECs I": "Early EECs",
    "Early ECs = Early ECs = Early EECs II": "Early ECs",
    "Early X cells = Early X cells = Early EECs IV": "Early X cells",
    
    # EEC subtypes
    "ECs = ECs ∋ Early ECs": "ECs",
    "ECs = ECs ∋ Late ECs": "ECs",
    "X cells = X cells = X cells": "X cells",
    "D cells = D cells = D cells": "D cells",
    "I/K cells = I/K cells ∋ Early I/K cells": "I/K cells",
    "I/K cells = I/K cells ∋ Late I/K cells": "I/K cells",
    
    # Misc (currently unresolved / rare types)
    "PCSK5+ cells = PCSK5+ cells = NONE": "UNRESOLVED",
    "GAU1+ cells = GAU1+ cells = UNRESOLVED": "UNRESOLVED",
}

In [ ]:
adata.obs["cell_type_merged"] = adata.obs["low_hierarchy"].map(resolved_annotations)

In [ ]:
adata.obs[["low_hierarchy", "cell_type_merged"]].drop_duplicates()

In [ ]:
rsc.pp.neighbors(adata, use_rep="X_pca_harmony")
rsc.tl.leiden(adata, resolution=2.0, key_added="leiden_r2.0")

In [ ]:
rsc.get.anndata_to_CPU(adata)

In [ ]:
color = ["sample", "cell_type_merged", "leiden_r2.0"]

for emb in ["harmony"]:
    sc.pl.embedding(
        adata,
        basis=f"X_umap_{emb}",
        color=color,
        size=10,
        wspace=0.4,
        layer="lognorm",
        title=[f"{col} ({emb})" for col in color],
    )

In [ ]:
sc.pl.embedding(
    adata,
    basis="X_umap_harmony",
    color=["GHRL", "GIP", "CCK", "SST", "TPH1", "NTS", "NEUROG3", "APOA4", "LGR5", "AVIL", "KIT", "FABP1", "FCGBP"],
    size=10,
    wspace=0.4,
    layer="lognorm",
    legend_loc="on data",
    legend_fontsize=8,
)

In [ ]:
# Remove unaligned/unclear cell types
remove_cell_types = ["UNRESOLVED"]

adata = adata[~adata.obs["cell_type_merged"].isin(remove_cell_types)]

In [ ]:
color = ["sample", "cell_type_merged"]

for emb in ["harmony"]:
    sc.pl.embedding(
        adata,
        basis=f"X_umap_{emb}",
        color=color,
        size=10,
        wspace=0.4,
        title=[f"{col} ({emb})" for col in color],
    )

In [ ]:
sc.pl.embedding(
    adata,
    basis=f"X_umap_{emb}",
    color="leiden_r2.0",
    size=10,
    wspace=0.4,
    legend_loc="on data",
    title=[f"{col} ({emb})" for col in color],
)

In [ ]:
# Drop cluster 0
adata = adata[adata.obs["leiden_r2.0"] != "0"].copy()

In [ ]:
def subcluster(adata, cluster_key="leiden", parent_cluster="7", use_rep="X_pca", resolution=0.5):
    """
    Subcluster cells from one Leiden cluster and add refined labels back.
    
    Parameters
    ----------
    adata : AnnData
        Full AnnData object with global clustering.
    cluster_key : str
        Column in adata.obs that holds the global cluster IDs (e.g. "leiden").
    parent_cluster : str
        Cluster ID in cluster_key that should be subclustered (e.g. "7").
    use_rep : str
        Representation to use for subclustering (e.g. "X_pca", "X_umap", "salient_rep").
    resolution : float
        Resolution for Leiden subclustering.
    """
    # subset the cluster of interest
    subset = adata[adata.obs[cluster_key] == parent_cluster].copy()

    # neighbors + subclustering
    sc.pp.neighbors(subset, use_rep=use_rep)
    sc.tl.leiden(subset, resolution=resolution, key_added="leiden_sub")

    # create refined labels like "7_0", "7_1"
    refined = [f"{parent_cluster}_{sub}" for sub in subset.obs["leiden_sub"]]
    subset.obs["refined"] = refined

    # add back into main AnnData
    adata.obs[f"{cluster_key}_refined"] = adata.obs[cluster_key].astype(str)
    adata.obs.loc[subset.obs.index, f"{cluster_key}_refined"] = subset.obs["refined"]

    return adata


In [ ]:
adata = subcluster(adata, cluster_key="leiden_r2.0", parent_cluster="23", use_rep="X_pca_harmony", resolution=0.5)

In [ ]:
sc.pl.embedding(
    adata,
    basis=f"X_umap_{emb}",
    color="leiden_r2.0_refined",
    size=10,
    wspace=0.4,
    legend_loc="on data",
    title=[f"{col} ({emb})" for col in color],
)

## Annotate using snapseed

In [ ]:
import hnoca.snapseed as snap

In [ ]:
marker_dict = {
    "Stem Cells": ["LGR5", "SMOC2"],
    "TA Cells": ["OLFM4", "TLN2"],
    "Cycling Cells": ["MKI67", "TOP2A"],
    "Enterocytes": ["FABP1", "APOA4"],
    "BEST4+ Enterocytes": ["BEST4", "NOTCH2"],
    "Goblet Cells": ["FCGBP", "MUC2"],
    "EEC Progenitors": ["NEUROG3", "RFX3"],
    "EC Cells": ["TPH1", "CHGB"],
    "D Cells": ["SST", "HHEX"],
    "X Cells": ["GHRL", "MLN"],
    "K Cells": ["GIP", "PLAGL1"],
    "I/N Cells": ["CCK", "NTS"],
}

In [ ]:
res = snap.annotate(
    adata,
    marker_dict,
    group_name="leiden_r2.0_refined",
    layer="lognorm",
)

In [ ]:
adata.obs = adata.obs.reset_index().merge(res, on="leiden_r2.0_refined", how="left").set_index("index")

In [ ]:
sc.pl.embedding(
    adata,
    basis=f"X_umap_{emb}",
    color="class",
    size=10,
    wspace=0.4,
    legend_loc="on data",
    title=[f"{col} ({emb})" for col in color],
)

In [ ]:
adata.obs["final_annotation"] = adata.obs["class"].astype(str)

In [ ]:
# mapping of only the clusters you want to change
mapping = {
    "5": "TA Cells",
    "7": "Cycling Cells",
    "10": "Cycling Cells",
    "24": "X Cells",
    "23_1": "I/N Cells",
    "23_3": "I/N Cells",
    "23_2": "K Cells",
    "23_4": "K Cells",
    "13": "Stem Cells",
}

# update only those entries
adata.obs.loc[
    adata.obs["leiden_r2.0_refined"].astype(str).isin(mapping.keys()), "final_annotation"
] = adata.obs["leiden_r2.0_refined"].astype(str).map(mapping)

In [ ]:
sc.pl.embedding(
    adata,
    basis=f"X_umap_{emb}",
    color="final_annotation",
    size=10,
    wspace=0.4,
    legend_loc="on data",
    title=[f"{col} ({emb})" for col in color],
)

In [ ]:
adata.obs

In [ ]:
adata.obs.drop(columns=["leiden_r2.0", "leiden_r2.0_refined", "class", "score", "auc", "expr", "low_hierarchy", "high_hierarchy", "cell_type_merged"], inplace=True)

In [ ]:
adata.write(os.path.join(data_dir, "10X.multiome.processed.controls.integrated.h5ad"))

## 07. Load integrated data

In [ ]:
sc.settings.set_figure_params(
    dpi=150,
    dpi_save=300,
    format="pdf",
    facecolor="white",
    frameon=False,
    vector_friendly=True,
    fontsize=10,
    figsize=(4, 4),
    transparent=True,
)

mpl.rcParams.update(
    {
        "svg.fonttype": "none",
        "pdf.fonttype": 42,
        "legend.fontsize": 6,
        "axes.titlesize": 6,
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
    }
)


In [ ]:
adata = sc.read(os.path.join(data_dir, "10X.multiome.processed.controls.integrated.h5ad"))

In [ ]:
ct_colors = {
    # Proliferative axis
    "Stem Cells": "#1f77b4",         # deep blue
    "TA Cells": "#4c91c6",           # medium blue
    "Cycling Cells": "#7aaed6",   # light blue
    
    # Absorptive lineage
    "Enterocytes": "#2ca02c",        # green
    "BEST4+ Enterocytes": "#1abc9c",       # distinct teal
    
    # Secretory lineage
    "Goblet Cells": "#ff7f0e",       # orange
    
    # EEC progenitors
    "EEC Progenitors": "#9467bd",         # purple
    
    # EEC subtypes
    "EC Cells": "#d62728",                # strong red
    "D Cells": "#17becf",            # cyan
    "X Cells": "#e377c2",            # magenta
    "I/N Cells": "#7f7f7f",            # dark grey
    "K Cells": "#bcbd22",          # olive
}


In [ ]:
sc.pl.embedding(
    adata,
    basis=f"X_umap_harmony",
    color="final_annotation",
    palette=ct_colors,
    size=3,
    wspace=0.35,
    title=f"final_annotation (harmony)",
    frameon=False,
    ncols=1,
    save=f"_multiome_controls_integrated_umap_harmony.pdf"
)

In [ ]:
adata.obs["final_annotation"] = adata.obs["final_annotation"].cat.reorder_categories([
    # Proliferative axis
    "Stem Cells",
    "TA Cells",
    "Cycling Cells",
    # Absorptive lineage
    "Enterocytes",
    "BEST4+ Enterocytes",
    # Secretory lineage
    "Goblet Cells",
    # EEC progenitors
    "EEC Progenitors",
    # EEC subtypes
    "EC Cells",
    "X Cells",
    "D Cells",
    "I/N Cells",
    "K Cells",
])

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=[
        "LGR5", "SMOC2",          # Stem Cells
        "OLFM4", "TLN2",         # TA Cells
        "MKI67", "TOP2A",        # Cycling Cells
        "FABP1", "APOA4",        # Enterocytes
        "BEST4", "NOTCH2",       # BEST4+ Enterocytes
        "FCGBP", "MUC2",         # Goblet Cells
        "NEUROG3", "RFX3",       # EEC Progenitors
        "TPH1", "CHGB",          # EC Cells
        "GHRL", "MLN",           # X Cells
        "SST", "HHEX",           # D Cells
        "CCK", "NTS",             # I/N Cells
        "GIP", "RFX6",         # K Cells
    ],
    groupby="final_annotation",
    standard_scale="var",
    cmap="Reds",
    swap_axes=True,
    figsize=(6, 6),
    show=True,
    save="_multiome_controls_integrated_marker_dotplot.pdf"
)

In [ ]:
adata.obs.to_csv(os.path.join(data_dir, "10X.multiome.processed.controls.integrated.cell_metadata.csv"))

In [ ]:
os.path.join(data_dir, "10X.multiome.processed.controls.integrated.cell_metadata.csv")

## Save integrated embedding

In [ ]:
base_dir = Path("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/atac/archr")

In [ ]:
adata = sc.read(os.path.join(data_dir, "10X.multiome.processed.controls.integrated.h5ad"))

In [ ]:
adata = adata[adata.obs["final_annotation"].isin(["EEC Progenitors", "EC Cells", "D Cells", "I/N Cells", "X Cells", "K Cells"])]

In [ ]:
# PCA Harmony
pd.DataFrame(
    adata.obsm["X_pca_harmony"],
    index=adata.obs_names
).to_csv(base_dir / "X_pca_harmony.tsv", sep="\t")

# UMAP Harmony
pd.DataFrame(
    adata.obsm["X_umap_harmony"],
    index=adata.obs_names
).to_csv(base_dir / "X_umap_harmony.tsv", sep="\t")

In [ ]:
adata.obs.to_csv(base_dir / "cell_metadata.tsv", sep="\t")

In [ ]:
# Write mtx, barcodes and features files for summarized experiment
from scipy import sparse
from scipy.io import mmwrite

mmwrite(base_dir / "gene_exp_matrix.mtx", sparse.csr_matrix(adata.layers["counts"]))
pd.DataFrame(adata.obs_names).to_csv(base_dir / "gene_exp_barcodes.tsv", sep="\t", header=False, index=False)
pd.DataFrame(adata.var_names).to_csv(base_dir / "gene_exp_features.tsv", sep="\t", header=False, index=False)

## Load ArchR gene score matrix

In [ ]:
mtx = sc.read(base_dir / "Multiome_v3GeneScoreMatrixSparse.mtx")
genes = pd.read_csv(base_dir / "Multiome_v3GeneNames.tsv", sep="\t", header=None, names=["gene_name"]).set_index("gene_name")
genes.set_index("gene_name")
cell_metadata = pd.read_csv(base_dir / "Multiome_v3CellMetaData.tsv", sep="\t")

In [ ]:
adata = ad.AnnData(X=mtx.X.T, obs=cell_metadata, var=genes)

In [ ]:
adata.obs["CellType"] = adata.obs["CellType"].astype("category").cat.reorder_categories([
    # Proliferative axis
    "Stem Cells",
    "TA Cells",
    "Cycling Cells",
    # Absorptive lineage
    "Enterocytes",
    "BEST4+ Enterocytes",
    # Secretory lineage
    "Goblet Cells",
    # EEC progenitors
    "EEC Progenitors",
    # EEC subtypes
    "EC Cells",
    "X Cells",
    "D Cells",
    "I/N Cells",
    "K Cells",
])

In [ ]:
sc.pl.dotplot(
    adata,
    var_names=[
        "LGR5", "SMOC2",          # Stem Cells
        "OLFM4", "TLN2",         # TA Cells
        "MKI67", "TOP2A",        # Cycling Cells
        "FABP1", "APOA4",        # Enterocytes
        "BEST4", "NOTCH2",       # BEST4+ Enterocytes
        "FCGBP", "MUC2",         # Goblet Cells
        "NEUROG3", "RFX3",       # EEC Progenitors
        "TPH1", "CHGB",          # EC Cells
        "GHRL", "MLN",           # X Cells
        "SST", "HHEX",           # D Cells
        "CCK", "NTS",             # I/N Cells
        "GIP", "RFX6",         # K Cells
    ],
    groupby="CellType",
    standard_scale="var",
    cmap="Purples",
    swap_axes=True,
    figsize=(6, 6),
    show=True,
    save="_multiome_controls_integrated_atac_gene_scores_marker_dotplot.pdf"
)

## Plot chromVAR scores

In [ ]:
motif_mat = pd.read_csv("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/atac/archr/motif_accessibility_matrix.csv")

In [ ]:
motif_mat = motif_mat.set_index(motif_mat.columns[0])

In [ ]:
# Export motif accessibility values as matrix
motif_vals <- getMatrixFromProject(proj, useMatrix = "MotifMatrix")
motif_mat <- assays(motif_vals)[["z"]]
write.csv(as.data.frame(as.matrix(motif_mat)), file = "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/atac/archr/motif_accessibility_matrix.csv")
